# Week 2 — Multivariable Calculus and the Minimum-Variance Portfolio

> Part of the open-source teaching project **quant-math-roadmap**.
> For **education and research methodology only** — not investment advice; no result here represents a profitable or investable strategy.

## Learning objectives

- Compute the gradient and Hessian of a quadratic objective function.
- Derive equality-constrained optimization with a Lagrange multiplier.
- Implement and interpret the minimum-variance portfolio.
- Observe how noisy covariance estimates make weights unstable.

## Estimated study time

About 9–11 hours.

## Prerequisites

- Partial derivatives and gradients
- Covariance and quadratic forms from Week 1

## External resources

- [MIT OpenCourseWare 18.02SC Multivariable Calculus](https://ocw.mit.edu/courses/18-02sc-multivariable-calculus-fall-2010/)
- [NTU OpenCourseWare: Foundations of Financial Literacy](https://ocw.aca.ntu.edu.tw/courses/110S204)

> External resources are linked for reference only; this project does not reproduce any copyrighted course material.

In [ ]:
# Teaching style setup (deterministic look, consistent figures)
import matplotlib as _mpl
_mpl.rcParams['axes.unicode_minus'] = False
_mpl.rcParams['figure.figsize'] = (8.5, 4.5)
_mpl.rcParams['savefig.dpi'] = 100
import numpy as _np
_np.random.seed(0)  # belt-and-braces; library functions take explicit seeds

## Concepts

### Gradient and Hessian

For $f(w) = w^\top\Sigma w$ (with $\Sigma$ symmetric), we have

$$ \nabla f(w) = 2\Sigma w, \qquad \nabla^2 f(w) = 2\Sigma. $$

If $\Sigma$ is PSD, the Hessian is PSD and $f$ is **convex** — which guarantees the minimization problem has a well-behaved, unique solution.

### The minimum-variance portfolio

The problem is:

$$ \min_w\ w^\top\Sigma w \quad \text{s.t.}\quad \mathbf{1}^\top w = 1. $$

Using the Lagrangian $L(w,\lambda) = w^\top\Sigma w - \lambda(\mathbf{1}^\top w - 1)$, differentiating with respect to $w$ and setting it to zero gives the closed-form solution

$$ w^\* = \frac{\Sigma^{-1}\mathbf{1}}{\mathbf{1}^\top\Sigma^{-1}\mathbf{1}}. $$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from quant_math_roadmap.data import SyntheticConfig, generate_correlated_returns
from quant_math_roadmap.finance.metrics import covariance_matrix
from quant_math_roadmap.finance.portfolio import (
    equal_weights, minimum_variance_portfolio,
    portfolio_variance, shrinkage_covariance,
)
from quant_math_roadmap.math.optimization import (
    quadratic_gradient, quadratic_hessian,
)

config = SyntheticConfig(n_assets=6, n_periods=504, seed=7,
                         average_correlation=0.45)
returns = generate_correlated_returns(config)
cov = covariance_matrix(returns).to_numpy()
cov.shape

### Numerical vs analytic gradient

In [ ]:
w0 = equal_weights(6)
analytic = quadratic_gradient(cov, w0)

# Verify the analytic gradient with finite differences
eps = 1e-6
numeric = np.zeros_like(w0)
for i in range(len(w0)):
    step = np.zeros_like(w0)
    step[i] = eps
    f_plus = portfolio_variance(w0 + step, cov)
    f_minus = portfolio_variance(w0 - step, cov)
    numeric[i] = (f_plus - f_minus) / (2 * eps)

print('Analytic gradient:', np.round(analytic, 6))
print('Numerical gradient:', np.round(numeric, 6))
print('Max error:', np.max(np.abs(analytic - numeric)))

The analytic gradient $2\Sigma w$ agrees closely with the finite differences. The Hessian is the constant matrix $2\Sigma$:

In [ ]:
hessian = quadratic_hessian(cov)
print('Is the Hessian symmetric?', np.allclose(hessian, hessian.T))
print('Smallest Hessian eigenvalue:', np.linalg.eigvalsh(hessian).min())
print('-> Non-negative eigenvalues mean the objective is convex.')

### Second-order Taylor approximation: what the function looks like *near* a point

For a quadratic objective, the second-order Taylor approximation is **exactly** equal to the original function at any expansion point (because the function itself is quadratic). Below we expand around the equal-weight point and verify that $f(w_0) + g^\top (w-w_0) + \tfrac12 (w-w_0)^\top H (w-w_0)$ matches the true value.

In [ ]:
from quant_math_roadmap.math.optimization import taylor_quadratic_approximation

w0 = equal_weights(6)
f0 = portfolio_variance(w0, cov)
g0 = quadratic_gradient(cov, w0)
H0 = quadratic_hessian(cov)
# Pick an offset in one direction as the point to approximate
w_far = w0 + np.array([0.1, -0.05, 0.0, -0.05, 0.0, 0.0])
approx = taylor_quadratic_approximation(f0, g0, H0, w0, w_far)
true = portfolio_variance(w_far, cov)
print(f'Second-order Taylor approximation = {approx:.10f}')
print(f'True value                        = {true:.10f}')
print('For a quadratic function the second-order Taylor approximation = the true value (zero error).')

### Minimum variance vs equal weight

In [ ]:
mvp = minimum_variance_portfolio(cov)
eq = equal_weights(6)
print('Minimum-variance weights:', np.round(mvp, 4), ' sum =', round(mvp.sum(), 6))
print('Equal weights           :', np.round(eq, 4))
print()
print(f'Minimum-variance portfolio variance = {portfolio_variance(mvp, cov):.8f}')
print(f'Equal-weight variance               = {portfolio_variance(eq, cov):.8f}')

**In-sample**, the minimum-variance portfolio's variance is by definition $\le$ the equal-weight one. But that does not guarantee it is better **out-of-sample** — which we test next.

### In-sample vs out-of-sample: instability caused by noise

In [ ]:
# Estimate weights on the first half, test on the second half
half = len(returns) // 2
train, test = returns.iloc[:half], returns.iloc[half:]
cov_train = covariance_matrix(train).to_numpy()
cov_test = covariance_matrix(test).to_numpy()

mvp_train = minimum_variance_portfolio(cov_train)
in_sample = portfolio_variance(mvp_train, cov_train)
out_sample = portfolio_variance(mvp_train, cov_test)
eq_out = portfolio_variance(eq, cov_test)
print(f'Minimum-variance in-sample variance      = {in_sample:.8f}')
print(f'Minimum-variance out-of-sample variance  = {out_sample:.8f}')
print(f'Equal-weight out-of-sample variance      = {eq_out:.8f}')
print('Observation: out-of-sample is usually worse than in-sample, sometimes even losing to equal weight.')

### Shrinkage: covariance estimation that fights noise

In [ ]:
weights_by_shrinkage = {}
for delta in [0.0, 0.2, 0.5, 0.8]:
    cov_shrunk = shrinkage_covariance(train, shrinkage=delta).to_numpy()
    w = minimum_variance_portfolio(cov_shrunk)
    weights_by_shrinkage[delta] = w

fig, ax = plt.subplots(figsize=(8, 4.5))
for delta, w in weights_by_shrinkage.items():
    ax.plot(range(1, 7), w, marker='o', label=f'shrinkage={delta}')
ax.axhline(1 / 6, linestyle='--', label='equal weight')
ax.set_title('Effect of shrinkage intensity on minimum-variance weights')
ax.set_xlabel('Asset index')
ax.set_ylabel('Weight')
ax.legend()
plt.show()

The stronger the shrinkage, the more the weights are pulled toward equal weight and away from extremes. This trades a little bias for a lot of stability, and often improves out-of-sample performance.

### Ledoit–Wolf: letting the data choose the shrinkage intensity

Above we tried a few shrinkage intensities by hand — but "how strong should it be" is itself an estimation problem. Ledoit & Wolf (2004) derived the optimal intensity that **minimizes the expected estimation error**, and it can be computed directly from the data. `ledoit_wolf_covariance()` wraps the scikit-learn implementation and returns the covariance matrix together with the data-driven shrinkage coefficient.

In [ ]:
from quant_math_roadmap.finance.portfolio import ledoit_wolf_covariance

lw_cov, lw_shrinkage = ledoit_wolf_covariance(train)
print(f'Shrinkage intensity chosen automatically by Ledoit-Wolf = {lw_shrinkage:.4f}')

w_lw = minimum_variance_portfolio(lw_cov.to_numpy())
out_lw = portfolio_variance(w_lw, cov_test)
print(f'Sample-covariance MVP out-of-sample variance = {out_sample:.8f}')
print(f'Ledoit-Wolf MVP out-of-sample variance       = {out_lw:.8f}')
print(f'Equal-weight out-of-sample variance          = {eq_out:.8f}')

Ledoit–Wolf needs no manual tuning, yet automatically keeps the weights only as extreme as the data can support. With many assets and relatively short samples (the norm in quant research), it is usually a more robust default than the raw sample covariance.

## Exercises

Work through these in order. **Basic exercises** consolidate the definitions, **applied exercises** are hands-on coding, and the **reflection question** connects the mathematics to backtesting and research methodology.

> The main notebook ships runnable starter code for each coding exercise. Full reference answers live in the matching `_solution` notebook under `notebooks/en/solutions/`.

### Basic exercises

1. Write down the Lagrangian of the minimum-variance problem and explain what each term means.
2. Why does "the Hessian is PSD" guarantee the problem has a well-behaved solution?
3. Explain in one sentence what trade-off shrinkage is making.

### Applied exercises

In [ ]:
# Applied exercise 1: compute the minimum-variance weights yourself using the closed form
# w* = (Σ^-1 1)/(1^T Σ^-1 1), and compare against minimum_variance_portfolio().
ones = np.ones(6)
my_mvp = None  # TODO: inv = np.linalg.solve(cov, ones); my_mvp = inv / (ones @ inv)
if my_mvp is not None:
    print('Max error:', np.max(np.abs(my_mvp - minimum_variance_portfolio(cov))))

In [ ]:
# Applied exercise 2: compute the long-only (no short selling) minimum-variance portfolio,
# and confirm there are no negative weights.
long_only = None  # TODO: minimum_variance_portfolio(cov, long_only=True)
if long_only is not None:
    print('Smallest weight:', long_only.min(), '| sum:', long_only.sum())

### Reflection question

1. The minimum-variance portfolio always beats equal weight in-sample, but not necessarily out-of-sample. How does this phenomenon relate to "overfitting the in-sample period" in Week 8?

## Quiz (self-check)
Answer the multiple-choice questions, then run the next cell to check yourself. Answers are stored as hashes, not plaintext.

**Q1. What is the closed-form solution w* of the minimum-variance portfolio?**
- A. Σ1 / (1ᵀΣ1)
- B. Σ⁻¹1 / (1ᵀΣ⁻¹1)
- C. 1/n equal weights
- D. Σ⁻¹μ

**Q2. What is the gradient of f(w) = wᵀΣw?**
- A. Σw
- B. 2Σw
- C. wᵀΣ
- D. 2w

**Q3. A PSD Hessian means the objective function has which property?**
- A. Convex
- B. Concave
- C. Linear
- D. Periodic

**Q4. What is the most common consequence of directly inverting a very noisy covariance estimate?**
- A. The code raises an error
- B. Extreme and unstable portfolio weights
- C. Higher returns
- D. Negative variance

In [ ]:
my_answers = {1: None, 2: None, 3: None, 4: None}  # TODO: fill in 'A' / 'B' / 'C' / 'D'

import hashlib as _hashlib
_expected = {1: '180e9f97a93bd901', 2: '1d515ad6574917a7', 3: '8dfd7f7d6dbc2621', 4: 'c0c97dcf7c115303'}
_n_correct = 0
for _q, _ans in my_answers.items():
    if _ans is None:
        print(f'Q{_q}: unanswered')
        continue
    _h = _hashlib.sha256(f'qmr-w2-q{_q}-{str(_ans).strip().upper()}'.encode()).hexdigest()[:16]
    _ok = _h == _expected[_q]
    _n_correct += int(_ok)
    print(f'Q{_q}: ' + ('✔ correct' if _ok else '✘ incorrect'))
print(f'Score: {_n_correct} / {len(my_answers)}')

## Common mistakes

- **Inverting a very noisy covariance matrix and getting extreme, unstable weights.**
- **Mistaking low in-sample variance for an out-of-sample guarantee.**
- **Forgetting the constraint $\mathbf{1}^\top w = 1$ and getting meaningless weights.**
- **Ignoring how sensitive the optimization result is to covariance estimation error.**

## After this week, you should be able to

- [ ] Write down and explain the Lagrangian of the minimum-variance problem.
- [ ] Compute the minimum-variance weights with the closed-form solution.
- [ ] Compare in-sample and out-of-sample variance.
- [ ] Explain how shrinkage stabilizes the weights.

## References and attribution

- Every explanation, example and exercise in this notebook is **original** to this project.
- Recommended external resources: [`docs/resources.md`](../../docs/resources.md).
- Concept notes: [`docs/math/`](../../docs/math/) and [`docs/finance/`](../../docs/finance/).

### Privacy and disclaimer

- This notebook contains no real personal information.
- This notebook uses only reproducible synthetic data and needs no network access.
- This notebook makes no claim of real-world trading profitability.